# LeetCode 2096 — Step-By-Step Directions From a Binary Tree Node to Another bằng Python

Link đề: https://leetcode.com/problems/step-by-step-directions-from-a-binary-tree-node-to-another/ · Độ khó: **Medium** · Chủ đề: String, Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def getDirections(self, root: Optional[TreeNode], startValue: int, destValue: int) -> str:
        ...
```

Lộ trình:
1. **BFS trên đồ thị vô hướng** (thêm cạnh tới cha), lưu "bước đi" để dựng lại đường — `O(n)`.
2. **Tìm LCA (tổ tiên chung thấp nhất)** rồi ghép đường `s → LCA` (toàn `U`) và `LCA → t`.
3. **Đường từ gốc tới hai nút + bỏ tiền tố chung** — gọn nhất (**nộp bài**).
4. **Con trỏ cha + độ sâu**: leo từ hai nút lên cho tới khi gặp nhau.

Mọi duyệt cây đều viết **lặp** — cây suy biến `10⁵` nút sẽ làm tràn giới hạn đệ quy của Python.

## 1. Mục tiêu học

1. Tự dựng `TreeNode` và hàm dựng cây từ mảng kiểu LeetCode (`[5,1,2,3,null,6,4]`).
2. Hiểu đường đi ngắn nhất giữa hai nút trong cây **duy nhất** và luôn đi qua **LCA**: đi lên tới LCA rồi đi xuống.
3. Tìm đường từ gốc tới một nút bằng DFS **lặp**, biểu diễn bằng chuỗi `L`/`R`.
4. Dùng nhận xét "**tiền tố chung** của hai chuỗi đường = đường từ gốc tới LCA" để giải gọn.
5. Biết các cách khác (BFS, con trỏ cha) và khi nào cần LCA nhanh cho nhiều truy vấn (binary lifting).

## 2. Đề bài & ví dụ

Cho cây nhị phân `n` nút, giá trị các nút là `1..n` (phân biệt). Cho `startValue` (nút `s`) và `destValue` (nút `t`, khác `s`). Tìm đường **ngắn nhất** từ `s` tới `t` và trả về chuỗi chỉ dẫn gồm:
- `'L'`: đi xuống con trái; `'R'`: đi xuống con phải; `'U'`: đi lên cha.

**Ví dụ 1:** `root = [5,1,2,3,null,6,4]`, `startValue = 3`, `destValue = 6` → `"UURL"` (`3 → 1 → 5 → 2 → 6`).

**Ví dụ 2:** `root = [2,1]`, `startValue = 2`, `destValue = 1` → `"L"`.

**Ràng buộc:** `2 ≤ n ≤ 10⁵`, giá trị phân biệt trong `1..n`, `startValue ≠ destValue`.

```
Ví dụ 1:        5
              /   \
             1     2
            /     / \
           3     6   4
```

In [1]:
import random
import sys
import time
from collections import deque
from typing import List, Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0]); q = deque([root]); i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root

def to_level(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val); q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

examples = [(([5, 1, 2, 3, None, 6, 4], 3, 6), "UURL"), (([2, 1], 2, 1), "L")]
for (arr, s, t), want in examples:
    print(arr, s, t, "->", want, "| dựng lại:", to_level(build_tree(arr)))

[5, 1, 2, 3, None, 6, 4] 3 6 -> UURL | dựng lại: [5, 1, 2, 3, None, 6, 4]
[2, 1] 2 1 -> L | dựng lại: [2, 1]


## 3. Kiến thức nền

### 3.1 DFS lặp tìm đường từ gốc tới một nút

Dùng ngăn xếp lưu `(nút, chuỗi_đường)`. Để tránh tạo chuỗi mới ở mỗi nút (tốn `O(n · độ sâu)`), ta lưu **cha** và **hướng** của mỗi nút, khi tìm thấy thì truy ngược.

In [2]:
def parents_of(root):
    """Trả về dict: val -> (val_cha, hướng 'L'/'R'); gốc có cha None."""
    par = {root.val: (None, "")}
    st = [root]
    while st:
        node = st.pop()
        for child, d in ((node.left, "L"), (node.right, "R")):
            if child:
                par[child.val] = (node.val, d)
                st.append(child)
    return par

def path_from_root(par, val):
    steps = []
    while par[val][0] is not None:
        p, d = par[val]
        steps.append(d); val = p
    return "".join(reversed(steps))

root = build_tree([5, 1, 2, 3, None, 6, 4])
par = parents_of(root)
for v in (5, 1, 2, 3, 6, 4):
    print(v, "->", repr(path_from_root(par, v)))

5 -> ''
1 -> 'L'
2 -> 'R'
3 -> 'LL'
6 -> 'RL'
4 -> 'RR'


### 3.2 Tiền tố chung dài nhất của hai chuỗi

In [3]:
def common_prefix_len(a, b):
    k = 0
    while k < len(a) and k < len(b) and a[k] == b[k]:
        k += 1
    return k

print(common_prefix_len("LL", "RL"), common_prefix_len("RL", "RR"), common_prefix_len("L", "LL"))

0 1 1


### 3.3 Giới hạn đệ quy của Python

Mặc định khoảng `1000` tầng. Cây "dây xích" (mọi nút chỉ có con trái) `10⁵` nút có độ sâu `10⁵` ⇒ DFS đệ quy sẽ `RecursionError`. Vì vậy mọi hàm trong notebook đều viết lặp.

In [4]:
print("recursion limit =", sys.getrecursionlimit())
def chain_tree(n):
    root = TreeNode(1); cur = root
    for v in range(2, n + 1):
        cur.left = TreeNode(v); cur = cur.left
    return root
deep = chain_tree(5000)
print("độ sâu cây dây xích 5000 nút: đường tới nút 5000 dài", len(path_from_root(parents_of(deep), 5000)))

recursion limit = 1000
độ sâu cây dây xích 5000 nút: đường tới nút 5000 dài 4999


## 4. Ý tưởng nền tảng / trực giác

Trong cây, giữa hai nút có **đúng một** đường đi đơn. Nó đi lên từ `s` tới **LCA** (tổ tiên chung thấp nhất) rồi đi xuống tới `t`.

Ví dụ 1: đường từ gốc
- tới `s = 3`: `"LL"` (`5 → 1 → 3`)
- tới `t = 6`: `"RL"` (`5 → 2 → 6`)

Tiền tố chung: `""` ⇒ LCA là gốc `5`. Từ `s` đi lên `len("LL") = 2` lần (`"UU"`), rồi đi theo `"RL"` ⇒ `"UURL"` ✓.

Một ví dụ khác cùng cây: `s = 6`, `t = 4`: đường `"RL"` và `"RR"`, tiền tố chung `"R"` (LCA = `2`) ⇒ `"U" + "R" = "UR"`.

| `s` | `t` | đường gốc→s | đường gốc→t | tiền tố chung | đáp án |
|---|---|---|---|---|---|
| 3 | 6 | LL | RL | — | UURL |
| 6 | 4 | RL | RR | R | UR |
| 1 | 3 | L | LL | L | L |
| 3 | 1 | LL | L | L | U |
| 4 | 5 | RR | — | — | UU |

In [5]:
for s, t in [(3, 6), (6, 4), (1, 3), (3, 1), (4, 5)]:
    ps, pt = path_from_root(par, s), path_from_root(par, t)
    k = common_prefix_len(ps, pt)
    print(f"s = {s}, t = {t}: '{ps}' vs '{pt}', tiền tố chung '{ps[:k]}' -> {'U' * (len(ps) - k) + pt[k:]!r}")

s = 3, t = 6: 'LL' vs 'RL', tiền tố chung '' -> 'UURL'
s = 6, t = 4: 'RL' vs 'RR', tiền tố chung 'R' -> 'UR'
s = 1, t = 3: 'L' vs 'LL', tiền tố chung 'L' -> 'L'
s = 3, t = 1: 'LL' vs 'L', tiền tố chung 'L' -> 'U'
s = 4, t = 5: 'RR' vs '', tiền tố chung '' -> 'UU'


## 5. Cách 1 — Brute Force / baseline: BFS trên đồ thị vô hướng

**Ý tưởng:** coi cây là đồ thị vô hướng — mỗi nút nối với con trái (`L`), con phải (`R`) và cha (`U`). BFS từ `s`, lưu cho mỗi nút "đến từ đâu, bằng bước gì"; tới `t` thì truy ngược. BFS cho đường ngắn nhất (và trong cây nó là đường duy nhất).

In [6]:
def directions_bfs(root, startValue, destValue):
    # dựng danh sách kề có nhãn
    adj = {}
    st = [root]
    while st:
        node = st.pop()
        adj.setdefault(node.val, [])
        for child, d in ((node.left, "L"), (node.right, "R")):
            if child:
                adj[node.val].append((child.val, d))
                adj.setdefault(child.val, []).append((node.val, "U"))
                st.append(child)
    prev = {startValue: (None, "")}
    q = deque([startValue])
    while q:
        u = q.popleft()
        if u == destValue:
            break
        for v, d in adj[u]:
            if v not in prev:
                prev[v] = (u, d); q.append(v)
    steps, v = [], destValue
    while prev[v][0] is not None:
        u, d = prev[v]; steps.append(d); v = u
    return "".join(reversed(steps))

for (arr, s, t), want in examples:
    print(directions_bfs(build_tree(arr), s, t), "kỳ vọng", want)

UURL kỳ vọng UURL
L kỳ vọng L


**Độ phức tạp:** `O(n)` thời gian và bộ nhớ (danh sách kề + `prev`). Không sai, chỉ "nặng tay": dựng cả đồ thị trong khi cấu trúc cây cho phép làm gọn hơn.

**Nút thắt:** BFS không khai thác việc mọi đường đều đi qua LCA.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi `P_s`, `P_t` là chuỗi `L/R` từ gốc tới `s`, `t`. Tiền tố chung dài nhất `C` của hai chuỗi chính là đường từ gốc tới **LCA** (hai đường trùng nhau cho tới khi tách nhánh). Khi đó:

`đáp án = 'U' × (|P_s| − |C|) + P_t[|C|:]`

**Vì sao so chuỗi `L/R` là đủ?** Trong cây, chuỗi hướng đi từ gốc xác định **duy nhất** một nút. Nên hai chuỗi trùng nhau ở `k` ký tự đầu ⇔ hai đường đi qua cùng `k + 1` nút đầu.

**Quy tắc:**
- *Nếu* `s` là tổ tiên của `t` *thì* `P_s` là tiền tố của `P_t` ⇒ không có `U`, chỉ đi xuống.
- *Nếu* `t` là tổ tiên của `s` *thì* chỉ toàn `U`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tìm LCA tường minh, rồi ghép hai đoạn

Tìm LCA bằng cách so hai danh sách **nút** trên đường từ gốc (dùng map cha), rồi: số `U` = độ sâu(s) − độ sâu(LCA); phần xuống = đường từ LCA tới `t`.

In [7]:
def directions_lca(root, startValue, destValue):
    par = parents_of(root)
    def ancestors(v):                       # danh sách nút từ v lên gốc
        out = [v]
        while par[v][0] is not None:
            v = par[v][0]; out.append(v)
        return out
    anc_s = ancestors(startValue)
    set_s = set(anc_s)
    # leo từ t lên tới khi gặp tổ tiên của s: đó là LCA
    down = []; v = destValue
    while v not in set_s:
        p, d = par[v]; down.append(d); v = p
    lca = v
    ups = anc_s.index(lca)                  # số bước từ s lên LCA
    return "U" * ups + "".join(reversed(down))

for (arr, s, t), want in examples:
    print(directions_lca(build_tree(arr), s, t), "kỳ vọng", want)

UURL kỳ vọng UURL
L kỳ vọng L


**Độ phức tạp:** `O(n)` để dựng map cha, `O(h)` cho phần còn lại (`h` = độ cao cây). Space `O(n)`.

### 7.2 Cách 3 — Hai đường từ gốc + bỏ tiền tố chung (code nộp)

Một lượt DFS lặp ghi cha/hướng, dựng `P_s`, `P_t`, cắt tiền tố chung.

In [8]:
def directions_prefix(root, startValue, destValue):
    par = parents_of(root)
    ps = path_from_root(par, startValue)
    pt = path_from_root(par, destValue)
    k = common_prefix_len(ps, pt)
    return "U" * (len(ps) - k) + pt[k:]

for (arr, s, t), want in examples:
    print(directions_prefix(build_tree(arr), s, t), "kỳ vọng", want)

UURL kỳ vọng UURL
L kỳ vọng L


Code nộp LeetCode của Cách 3: xem mục 9 (bản tối ưu dừng DFS khi đã gặp đủ hai nút).

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.3 Cách 4 — Con trỏ cha + độ sâu, leo đồng thời

Tính `depth` cho mọi nút. Đưa nút sâu hơn lên cho bằng độ sâu, rồi leo cả hai cùng lúc tới khi trùng — đó là LCA. Ghi các bước của phía `t` để đảo ngược làm đoạn đi xuống.

In [9]:
def directions_climb(root, startValue, destValue):
    par = {root.val: (None, "")}; depth = {root.val: 0}
    st = [root]
    while st:
        node = st.pop()
        for child, d in ((node.left, "L"), (node.right, "R")):
            if child:
                par[child.val] = (node.val, d); depth[child.val] = depth[node.val] + 1; st.append(child)
    a, b = startValue, destValue
    ups, down = 0, []
    while depth[a] > depth[b]:
        a = par[a][0]; ups += 1
    while depth[b] > depth[a]:
        down.append(par[b][1]); b = par[b][0]
    while a != b:
        a = par[a][0]; ups += 1
        down.append(par[b][1]); b = par[b][0]
    return "U" * ups + "".join(reversed(down))

for (arr, s, t), want in examples:
    print(directions_climb(build_tree(arr), s, t), "kỳ vọng", want)

UURL kỳ vọng UURL
L kỳ vọng L


**Độ phức tạp:** `O(n)` dựng, `O(h)` leo. Đây là nền tảng của **binary lifting** (nhảy `2^j` bước một lúc) khi có nhiều truy vấn (Bài 5).

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1: in quá trình DFS ghi cha, hai đường, tiền tố chung.

In [10]:
def trace_prefix(arr, s, t):
    root = build_tree(arr)
    par = {root.val: (None, "")}; st = [root]; order = []
    while st:
        node = st.pop(); order.append(node.val)
        for child, d in ((node.left, "L"), (node.right, "R")):
            if child:
                par[child.val] = (node.val, d); st.append(child)
    print("thứ tự lấy ra khỏi ngăn xếp:", order)
    print("cha/hướng:", {v: par[v] for v in sorted(par)})
    for name, v in (("s", s), ("t", t)):
        chain, x = [], v
        while par[x][0] is not None:
            chain.append(f"{par[x][0]}-{par[x][1]}->{x}"); x = par[x][0]
        print(f"  truy ngược {name} = {v}: {list(reversed(chain))} -> '{path_from_root(par, v)}'")
    ps, pt = path_from_root(par, s), path_from_root(par, t); k = common_prefix_len(ps, pt)
    print(f"  tiền tố chung dài {k}; số U = {len(ps) - k}; phần xuống = '{pt[k:]}' -> '{'U' * (len(ps) - k) + pt[k:]}'")

trace_prefix([5, 1, 2, 3, None, 6, 4], 3, 6)

thứ tự lấy ra khỏi ngăn xếp: [5, 2, 4, 6, 1, 3]
cha/hướng: {1: (5, 'L'), 2: (5, 'R'), 3: (1, 'L'), 4: (2, 'R'), 5: (None, ''), 6: (2, 'L')}
  truy ngược s = 3: ['5-L->1', '1-L->3'] -> 'LL'
  truy ngược t = 6: ['5-R->2', '2-L->6'] -> 'RL'
  tiền tố chung dài 0; số U = 2; phần xuống = 'RL' -> 'UURL'


In [11]:
def trace_climb(arr, s, t):
    root = build_tree(arr)
    par = {root.val: (None, "")}; depth = {root.val: 0}; st = [root]
    while st:
        node = st.pop()
        for child, d in ((node.left, "L"), (node.right, "R")):
            if child: par[child.val] = (node.val, d); depth[child.val] = depth[node.val] + 1; st.append(child)
    a, b, ups, down = s, t, 0, []
    print(f"bắt đầu a = {a} (sâu {depth[a]}), b = {b} (sâu {depth[b]})")
    while a != b:
        if depth[a] >= depth[b]:
            a = par[a][0]; ups += 1; print(f"  a lên -> {a}")
        else:
            down.append(par[b][1]); b = par[b][0]; print(f"  b lên -> {b} (ghi '{down[-1]}')")
    print(f"  gặp nhau ở LCA = {a}: {'U' * ups + ''.join(reversed(down))!r}")

trace_climb([1, 2, 3, 4, 5, 6, 7, None, None, 8, 9], 8, 6)

bắt đầu a = 8 (sâu 3), b = 6 (sâu 2)
  a lên -> 5
  a lên -> 2
  b lên -> 3 (ghi 'L')
  a lên -> 1
  b lên -> 1 (ghi 'R')
  gặp nhau ở LCA = 1: 'UUURL'


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def getDirections(self, root: Optional[TreeNode], startValue: int, destValue: int) -> str:
        par = {root.val: (None, "")}
        st = [root]
        found = 0
        while st and found < 2:
            node = st.pop()
            if node.val == startValue or node.val == destValue:
                found += 1
            for child, d in ((node.left, "L"), (node.right, "R")):
                if child:
                    par[child.val] = (node.val, d)
                    st.append(child)

        def path(v):
            steps = []
            while par[v][0] is not None:
                p, d = par[v]
                steps.append(d)
                v = p
            return steps[::-1]

        ps, pt = path(startValue), path(destValue)
        k = 0
        while k < len(ps) and k < len(pt) and ps[k] == pt[k]:
            k += 1
        return "U" * (len(ps) - k) + "".join(pt[k:])

sol = Solution()
for (arr, s, t), want in examples:
    got = sol.getDirections(build_tree(arr), s, t)
    print(got, "OK" if got == want else "SAI")

UURL OK
L OK


Chi tiết: một nút được ghi `par` **khi cha của nó được lấy ra** khỏi ngăn xếp, nên lúc `found == 2` (đã lấy ra cả `s` lẫn `t`) thì `par` của `s`, `t` và mọi tổ tiên của chúng đều đã có — dừng sớm an toàn.

## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đường duy nhất qua LCA).** Trong cây, đường đơn giữa `s` và `t` là duy nhất. Gọi `a = LCA(s, t)`. Đường `s → a` (đi lên) nối với `a → t` (đi xuống) là một đường đơn (hai đoạn chỉ chung đỉnh `a`, vì nếu chung đỉnh khác thì đó là tổ tiên chung thấp hơn `a`). Vậy đó là đường ngắn nhất.

**Bổ đề 2 (tiền tố chung = đường tới LCA).** Đường gốc→`s` và gốc→`t` cùng đi qua mọi tổ tiên chung và tách nhau ngay sau `a`. Do chuỗi `L/R` xác định nút duy nhất, tiền tố chung dài nhất của `P_s`, `P_t` là đúng `P_a`.

⇒ Đoạn lên có `|P_s| − |P_a|` bước `U`; đoạn xuống là `P_t` bỏ `|P_a|` ký tự đầu. ✓

**Cách 4.** Sau khi cân bằng độ sâu, `a` và `b` cùng độ sâu; leo đồng thời giữ cùng độ sâu, nên lần đầu `a = b` là tổ tiên chung **sâu nhất**. ✓

## 11. Phân tích Time / Space

`n ≤ 10⁵`, `h` = độ cao cây (`≤ n`).

| Cách | Time | Space |
|---|---|---|
| 1. BFS đồ thị | `O(n)` | `O(n)` |
| 2. LCA + ghép | `O(n)` | `O(n)` |
| 3. Hai đường + tiền tố | `O(n)` | `O(n)` |
| 4. Cha + độ sâu | `O(n)` dựng + `O(h)` | `O(n)` |

Đo trên cây ngẫu nhiên và cây dây xích `10⁵` nút (độ sâu `10⁵` — đệ quy sẽ không chạy nổi):

In [13]:
def random_tree(n, rng):
    vals = list(range(1, n + 1)); rng.shuffle(vals)
    nodes = [TreeNode(v) for v in vals]; slots = [(nodes[0], "L"), (nodes[0], "R")]
    for node in nodes[1:]:
        k = rng.randrange(len(slots)); p, side = slots[k]; slots[k] = slots[-1]; slots.pop()
        if side == "L": p.left = node
        else: p.right = node
        slots += [(node, "L"), (node, "R")]
    return nodes[0]

rng = random.Random(1)
trees = [("random", random_tree(10**5, rng), rng.randint(1, 10**5), rng.randint(1, 10**5)),
         ("chain", chain_tree(10**5), 10**5, 1)]
for tname, rt, s, t in trees:
    if s == t: t = s % 10**5 + 1
    res = []
    for name, fn in [("bfs", directions_bfs), ("lca", directions_lca), ("prefix", directions_prefix),
                     ("climb", directions_climb), ("submit", Solution().getDirections)]:
        t0 = time.perf_counter(); r = fn(rt, s, t); res.append(r)
        print(f"{tname:6s} {name:6s}: độ dài {len(r):>6}  {time.perf_counter() - t0:.3f}s")
    assert len(set(res)) == 1

random bfs   : độ dài     38  0.234s
random lca   : độ dài     38  0.055s
random prefix: độ dài     38  0.049s
random climb : độ dài     38  0.064s


random submit: độ dài     38  0.069s


chain  bfs   : độ dài  99999  0.330s
chain  lca   : độ dài  99999  0.034s
chain  prefix: độ dài  99999  0.029s
chain  climb : độ dài  99999  0.035s
chain  submit: độ dài  99999  0.032s


## 12. Bộ test

Đối chiếu với BFS (độc lập với ý tưởng LCA) và **mô phỏng chuỗi chỉ dẫn**: đi theo chuỗi từ `s` phải tới đúng `t`, và độ dài chuỗi bằng khoảng cách BFS.

In [14]:
fns = [directions_lca, directions_prefix, directions_climb, Solution().getDirections]

def follow(root, startValue, directions):
    par = {}; nodes = {}; st = [root]
    while st:
        node = st.pop(); nodes[node.val] = node
        for c in (node.left, node.right):
            if c: par[c.val] = node; st.append(c)
    cur = nodes[startValue]
    for ch in directions:
        cur = par[cur.val] if ch == "U" else (cur.left if ch == "L" else cur.right)
        if cur is None: return None
    return cur.val

for (arr, s, t), want in examples:
    assert directions_bfs(build_tree(arr), s, t) == want
    for fn in fns:
        assert fn(build_tree(arr), s, t) == want

edge = [(([1, 2], 2, 1), "U"), (([1, None, 2], 1, 2), "R"), (([1, 2, 3], 2, 3), "UR"), (([1, 2, 3], 3, 2), "UL"),
        (([1, 2, None, 3, None, 4], 4, 1), "UUU"), (([1, 2, None, 3, None, 4], 1, 4), "LLL")]
for (arr, s, t), want in edge:
    for fn in fns + [directions_bfs]:
        assert fn(build_tree(arr), s, t) == want, (fn, arr, s, t)

rng = random.Random(2024)
for _ in range(500):
    n = rng.randint(2, 25); rt = random_tree(n, rng)
    s, t = rng.sample(range(1, n + 1), 2)
    want = directions_bfs(rt, s, t)
    assert follow(rt, s, want) == t
    for fn in fns:
        got = fn(rt, s, t)
        assert got == want and follow(rt, s, got) == t, (fn, to_level(rt), s, t)
print("ví dụ + edge + 500 cây ngẫu nhiên: khớp BFS và đi theo chỉ dẫn tới đúng đích")

ví dụ + edge + 500 cây ngẫu nhiên: khớp BFS và đi theo chỉ dẫn tới đúng đích


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Luôn đi lên tới **gốc** (không bỏ tiền tố chung)

In [15]:
def wrong_via_root(root, s, t):
    par = parents_of(root)
    ps, pt = path_from_root(par, s), path_from_root(par, t)
    return "U" * len(ps) + pt                                   # SAI: không ngắn nhất khi LCA không phải gốc

rt = build_tree([5, 1, 2, 3, None, 6, 4])
for s, t in [(3, 6), (6, 4), (1, 3)]:
    print(f"s = {s}, t = {t}: sai = {wrong_via_root(rt, s, t)!r} | đúng = {Solution().getDirections(rt, s, t)!r}")

s = 3, t = 6: sai = 'UURL' | đúng = 'UURL'
s = 6, t = 4: sai = 'UURR' | đúng = 'UR'
s = 1, t = 3: sai = 'ULL' | đúng = 'L'


Đi qua gốc vẫn tới đích nhưng **không ngắn nhất** khi LCA nằm thấp hơn (`6 → 4` chỉ cần `"UR"`). Ví dụ 1 tình cờ đúng vì LCA là gốc. **Sửa:** cắt tiền tố chung.

### Lỗi 2 — Dùng chính đường `P_s` (đảo ngược) thay vì `U`

In [16]:
def wrong_reverse_letters(root, s, t):
    par = parents_of(root)
    ps, pt = path_from_root(par, s), path_from_root(par, t); k = common_prefix_len(ps, pt)
    return ps[k:][::-1] + pt[k:]                                # SAI: đi lên phải là 'U', không phải 'L'/'R'

for s, t in [(3, 6), (6, 4)]:
    print(f"s = {s}, t = {t}: sai = {wrong_reverse_letters(rt, s, t)!r} | đúng = {Solution().getDirections(rt, s, t)!r}")

s = 3, t = 6: sai = 'LLRL' | đúng = 'UURL'
s = 6, t = 4: sai = 'LR' | đúng = 'UR'


Đi **lên** luôn là `'U'` bất kể ta là con trái hay con phải. **Sửa:** `'U' * (len(P_s) − k)`.

### Lỗi 3 — Quên đảo ngược khi truy ngược từ nút lên gốc

In [17]:
def wrong_no_reverse(root, s, t):
    par = parents_of(root)
    def path_unreversed(v):
        steps = []
        while par[v][0] is not None:
            steps.append(par[v][1]); v = par[v][0]
        return "".join(steps)                                    # SAI: thứ tự từ nút lên gốc
    ps, pt = path_unreversed(s), path_unreversed(t); k = common_prefix_len(ps, pt)
    return "U" * (len(ps) - k) + pt[k:]

for s, t in [(3, 6), (6, 4), (5, 6)]:
    print(f"s = {s}, t = {t}: sai = {wrong_no_reverse(rt, s, t)!r} | đúng = {Solution().getDirections(rt, s, t)!r}")

s = 3, t = 6: sai = 'UR' | đúng = 'UURL'
s = 6, t = 4: sai = 'UURR' | đúng = 'UR'
s = 5, t = 6: sai = 'LR' | đúng = 'RL'


Truy ngược thu được chuỗi theo thứ tự **nút → gốc**; tiền tố chung phải so theo thứ tự **gốc → nút**. Lỗi này thường cho chuỗi có độ dài đúng nhưng hướng sai, hoặc cắt nhầm tiền tố (`"LR"` ngược lại thành `"RL"`). **Sửa:** đảo ngược.

### Lỗi 4 — DFS đệ quy trên cây sâu

In [18]:
def path_recursive(node, target, acc):
    if node is None: return None
    if node.val == target: return "".join(acc)
    for child, d in ((node.left, "L"), (node.right, "R")):
        acc.append(d)
        r = path_recursive(child, target, acc)
        if r is not None: return r
        acc.pop()
    return None

try:
    path_recursive(chain_tree(3000), 3000, [])
    print("chạy được")
except RecursionError as e:
    print("RecursionError trên cây dây xích 3000 nút:", str(e)[:45])

RecursionError trên cây dây xích 3000 nút: maximum recursion depth exceeded


**Sửa:** DFS lặp với ngăn xếp tường minh (mọi cách trong notebook). Tăng `sys.setrecursionlimit` cũng có thể gây tràn stack C thật sự — không an toàn.

### Lỗi 5 — Ghép chuỗi trong mỗi nút của DFS (`path + 'L'`)

Đúng nhưng `O(n · h)` về lý thuyết: với cây dây xích `n` nút, tổng độ dài các chuỗi tạo ra là `≈ n²/2` ký tự. Sao chép chuỗi trong CPython rất nhanh nên hằng số nhỏ, nhưng **bậc** vẫn là bình phương — đo bên dưới. **Sửa:** lưu cha/hướng rồi truy ngược một lần (`O(h)`), như mục 3.1.

In [19]:
def path_concat(root, target):
    st = [(root, "")]
    while st:
        node, p = st.pop()
        if node.val == target: return p
        if node.right: st.append((node.right, p + "R"))
        if node.left: st.append((node.left, p + "L"))

for n in (20000, 40000, 80000):
    rt_c = chain_tree(n)
    t0 = time.perf_counter(); path_concat(rt_c, n); t1 = time.perf_counter()
    path_from_root(parents_of(rt_c), n); t2 = time.perf_counter()
    print(f"n = {n}: ghép chuỗi {t1 - t0:.3f}s | truy ngược {t2 - t1:.3f}s")

n = 20000: ghép chuỗi 0.005s | truy ngược 0.008s
n = 40000: ghép chuỗi 0.018s | truy ngược 0.012s


n = 80000: ghép chuỗi 0.077s | truy ngược 0.019s


Khi `n` gấp đôi, thời gian ghép chuỗi tăng khoảng **4 lần** (dấu hiệu bình phương), còn truy ngược chỉ tăng khoảng 2 lần (tuyến tính). Với cây sâu hơn hoặc nhiều truy vấn, khác biệt này trở nên đáng kể.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đường giữa hai nút trong cây" ⇒ đi qua **LCA**: lên rồi xuống.
- Biểu diễn đường từ gốc bằng chuỗi hướng ⇒ LCA = **tiền tố chung**.
- Python + cây có thể suy biến ⇒ **DFS lặp**, lưu cha/hướng, truy ngược.
- Nhiều truy vấn LCA ⇒ binary lifting hoặc Euler tour + RMQ.
- Họ bài: 236 *Lowest Common Ancestor of a Binary Tree*, 1740 *Find Distance in a Binary Tree*, 863 *All Nodes Distance K in Binary Tree*, 1650 *LCA of a Binary Tree III* (có con trỏ cha).

## 15. Biến thể của bài

1. **Khoảng cách** giữa hai nút (1740): độ dài chuỗi = `|P_s| + |P_t| − 2·|C|`.
2. **Đường đi dạng danh sách giá trị nút** thay vì chỉ dẫn.
3. **Có con trỏ cha sẵn** (1650): không cần duyệt từ gốc — dùng Cách 4.
4. **Nhiều truy vấn** `(s, t)` trên cùng cây: binary lifting `O(log n)` mỗi LCA (Bài 5).
5. **Đồ thị tổng quát**: không còn LCA — phải BFS (Cách 1).

In [20]:
def distance(root, s, t):
    """Biến thể 1."""
    par = parents_of(root); ps, pt = path_from_root(par, s), path_from_root(par, t)
    k = common_prefix_len(ps, pt)
    return len(ps) + len(pt) - 2 * k

def path_values(root, s, t):
    """Biến thể 2: danh sách giá trị nút trên đường s -> t."""
    par = parents_of(root)
    anc_s = [s]
    while par[anc_s[-1]][0] is not None: anc_s.append(par[anc_s[-1]][0])
    pos = {v: i for i, v in enumerate(anc_s)}
    down = [t]
    while down[-1] not in pos: down.append(par[down[-1]][0])
    lca = down[-1]
    return anc_s[:pos[lca] + 1] + down[-2::-1]

rt = build_tree([5, 1, 2, 3, None, 6, 4])
print("khoảng cách 3 -> 6:", distance(rt, 3, 6), "| đường:", path_values(rt, 3, 6), "| 6 -> 4:", path_values(rt, 6, 4))
rng = random.Random(15)
for _ in range(300):
    n = rng.randint(2, 20); tr = random_tree(n, rng); s, t = rng.sample(range(1, n + 1), 2)
    assert distance(tr, s, t) == len(directions_bfs(tr, s, t))
    pv = path_values(tr, s, t)
    assert pv[0] == s and pv[-1] == t and len(pv) == distance(tr, s, t) + 1

khoảng cách 3 -> 6: 4 | đường: [3, 1, 5, 2, 6] | 6 -> 4: [6, 2, 4]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS đồ thị | coi cây là đồ thị | `O(n)` | `O(n)` | kiểm chứng; đồ thị tổng quát |
| 2. LCA + ghép | tập tổ tiên | `O(n)` | `O(n)` | khi cần LCA tường minh |
| 3. Hai đường + tiền tố | so chuỗi | `O(n)` | `O(n)` | **nộp bài**, gọn |
| 4. Cha + độ sâu | leo đồng thời | `O(n)` + `O(h)` | `O(n)` | có con trỏ cha; nền cho binary lifting |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** cây `[5,1,2,3,null,6,4]`: `s = 4, t = 3`? `s = 2, t = 6`? `s = 6, t = 5`?

**Bài 2 (tính tay):** cây `[1,2,3,4,5,6,7,null,null,8,9]`, `s = 8`, `t = 6`. Viết `P_s`, `P_t`, tiền tố chung và đáp án.

**Bài 3 (điền TODO):** hoàn thiện `apply_directions(root, startValue, directions)` trả về giá trị nút sau khi đi theo chuỗi chỉ dẫn (giả sử chuỗi hợp lệ).

**Bài 4 (tự viết lại):** viết `lca_value(root, p, q)` trả về giá trị LCA bằng tiền tố chung (không dùng đệ quy).

**Bài 5 (mini challenge):** cho cây và **nhiều** truy vấn `(s, t)`; trả về **độ dài** chỉ dẫn cho mỗi truy vấn trong `O(log n)` mỗi truy vấn sau tiền xử lý `O(n log n)` bằng binary lifting. Kiểm chứng bằng Cách 3. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [21]:
def apply_directions(root, startValue, directions):
    # TODO: dựng map cha + map giá trị -> nút; đi 'U' lên cha, 'L'/'R' xuống con
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `P_4 = "RR"`, `P_3 = "LL"`, `P_2 = "R"`, `P_6 = "RL"`, `P_5 = ""`. (2) Cắt tiền tố chung. (3) `"UULL"`, `"L"`, `"UU"`.

**Bài 2:** (1) `8` là con trái của `5`, `5` là con phải của `2` ⇒ `P_8 = "LRL"`. (2) `P_6 = "RL"`. (3) Không có tiền tố chung ⇒ `"UUU" + "RL"`.

**Bài 3:** (1) Như hàm `follow` ở mục 12. (2) `'U'` ⇒ `cur = par[cur]`. (3) Trả `cur.val`.

**Bài 4:** (1) Tính `P_p`, `P_q`, `k`. (2) Đi từ gốc theo `P_p[:k]`. (3) Nút đến được là LCA.

**Bài 5:** (1) `up[j][v]` = tổ tiên thứ `2^j` của `v`. (2) Đưa hai nút về cùng độ sâu bằng các bước `2^j`, rồi nhảy đồng thời từ `j` lớn xuống nhỏ khi `up[j][a] ≠ up[j][b]`. (3) Độ dài = `depth[s] + depth[t] − 2·depth[lca]`.

## 19. Đáp án tham khảo

In [22]:
rt = build_tree([5, 1, 2, 3, None, 6, 4])
# Bài 1
assert [Solution().getDirections(rt, s, t) for s, t in [(4, 3), (2, 6), (6, 5)]] == ["UULL", "L", "UU"]
# Bài 2
rt2 = build_tree([1, 2, 3, 4, 5, 6, 7, None, None, 8, 9])
assert Solution().getDirections(rt2, 8, 6) == "UUURL" == directions_bfs(rt2, 8, 6)

# Bài 3
def apply_directions_ans(root, startValue, directions):
    return follow(root, startValue, directions)

# Bài 4
def lca_value_ans(root, p, q):
    par = parents_of(root); pp, pq = path_from_root(par, p), path_from_root(par, q)
    k = common_prefix_len(pp, pq); cur = root
    for ch in pp[:k]:
        cur = cur.left if ch == "L" else cur.right
    return cur.val

def lca_brute(root, p, q):
    par = parents_of(root); anc = set()
    while p is not None: anc.add(p); p = par[p][0]
    while q not in anc: q = par[q][0]
    return q

rng = random.Random(41)
for _ in range(500):
    n = rng.randint(2, 25); tr = random_tree(n, rng); s, t = rng.sample(range(1, n + 1), 2)
    assert apply_directions_ans(tr, s, Solution().getDirections(tr, s, t)) == t
    assert lca_value_ans(tr, s, t) == lca_brute(tr, s, t)
print("Bài 4 ví dụ: LCA(3, 6) =", lca_value_ans(rt, 3, 6), "| LCA(6, 4) =", lca_value_ans(rt, 6, 4))
print("Bài 1-4: OK")

Bài 4 ví dụ: LCA(3, 6) = 5 | LCA(6, 4) = 2
Bài 1-4: OK


In [23]:
# Bài 5 — binary lifting cho nhiều truy vấn độ dài chỉ dẫn
class LiftingLCA:
    def __init__(self, root):
        par = {root.val: root.val}; depth = {root.val: 0}; st = [root]
        while st:
            node = st.pop()
            for c in (node.left, node.right):
                if c: par[c.val] = node.val; depth[c.val] = depth[node.val] + 1; st.append(c)
        self.depth = depth
        self.LOG = max(1, max(depth.values()).bit_length())
        self.up = [par]
        for j in range(1, self.LOG):
            prev = self.up[-1]
            self.up.append({v: prev[prev[v]] for v in prev})
    def lca(self, a, b):
        d = self.depth
        if d[a] < d[b]: a, b = b, a
        diff = d[a] - d[b]; j = 0
        while diff:
            if diff & 1: a = self.up[j][a]
            diff >>= 1; j += 1
        if a == b: return a
        for j in range(self.LOG - 1, -1, -1):
            if self.up[j][a] != self.up[j][b]:
                a, b = self.up[j][a], self.up[j][b]
        return self.up[0][a]
    def dist(self, a, b):
        return self.depth[a] + self.depth[b] - 2 * self.depth[self.lca(a, b)]

rng = random.Random(55)
for _ in range(100):
    n = rng.randint(2, 60); tr = random_tree(n, rng); L = LiftingLCA(tr)
    for _ in range(20):
        s, t = rng.sample(range(1, n + 1), 2)
        assert L.dist(s, t) == len(Solution().getDirections(tr, s, t))
big_t = random_tree(10**5, random.Random(9)); L = LiftingLCA(big_t)
qs = [tuple(random.Random(i).sample(range(1, 10**5 + 1), 2)) for i in range(2000)]
t0 = time.perf_counter(); r = [L.dist(s, t) for s, t in qs]
print(f"Bài 5: khớp Cách 3; 2000 truy vấn trên cây 1e5 nút: {time.perf_counter() - t0:.3f}s (độ dài TB {sum(r) / len(r):.1f})")

Bài 5: khớp Cách 3; 2000 truy vấn trên cây 1e5 nút: 0.007s (độ dài TB 36.5)


## 20. Template nhớ nhanh + Checklist

```python
# Chỉ dẫn s -> t trong cây nhị phân
par = {root.val: (None, "")}; st = [root]
while st:                                   # DFS lặp ghi cha + hướng
    node = st.pop()
    for child, d in ((node.left, "L"), (node.right, "R")):
        if child: par[child.val] = (node.val, d); st.append(child)

def path(v):                                # gốc -> v dạng 'L'/'R'
    s = []
    while par[v][0] is not None: s.append(par[v][1]); v = par[v][0]
    return s[::-1]

ps, pt = path(start), path(dest)
k = common_prefix_len(ps, pt)               # = đường gốc -> LCA
ans = "U" * (len(ps) - k) + "".join(pt[k:])
```

**Checklist trước khi submit:**
- [ ] Cắt **tiền tố chung** (đi qua LCA, không qua gốc).
- [ ] Đoạn đi lên là toàn `'U'`.
- [ ] Đảo ngược chuỗi khi truy ngược từ nút lên gốc.
- [ ] DFS **lặp** (cây có thể sâu `10⁵`).
- [ ] Không ghép chuỗi tại mỗi nút (`O(n·h)`).